Shipment Booking Prediction

Predict the next booking date and shipment type for each company based on historical booking data (2021–2025).

Approach:-

Next Booking Date - Weighted Moving Average of inter-booking gaps 

Shipment Type - Mode over a recent 90-day window 

In [23]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [24]:
df = pd.read_csv('shipment_booking_data_sparse_realistic_2021_2025.csv')
df.head()

,company_name,shipment_type,booking_date
0,BlueDart,Surface,2021-01-01
1,BlueDart,Surface,2021-01-01
2,BlueDart,Surface,2021-01-01
3,BlueDart,International,2021-01-01
4,BlueDart,Surface,2021-01-01


In [25]:
df['booking_date'] = pd.to_datetime(df['booking_date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128820 entries, 0 to 128819
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   company_name   128820 non-null  str           
 1   shipment_type  128820 non-null  str           
 2   booking_date   128820 non-null  datetime64[us]
dtypes: datetime64[us](1), str(2)
memory usage: 2.9 MB


In [26]:
print(f"Total Records : {len(df):}")
print(f"Companies     : {df['company_name'].unique()}")
print(f"Shipment Types: {sorted(df['shipment_type'].unique())}")
print(f"Date Range    : {df['booking_date'].min().date()} → {df['booking_date'].max().date()}")
print(f"Missing Values: {df.isnull().sum().sum()}")

Total Records : 128820
Companies     : <StringArray>
[    'BlueDart',    'Delhivery',         'DTDC',  'FedEx India',
  'DHL Express',   'XpressBees', 'Ecom Express',    'Shadowfax']
Length: 8, dtype: str
Shipment Types: ['Air', 'Express', 'International', 'Surface']
Date Range    : 2021-01-01 → 2025-12-31
Missing Values: 0


In [27]:
#Company wise booking count
df['company_name'].value_counts()

company_name
Delhivery       33454
BlueDart        31636
DTDC            20576
XpressBees      16566
Ecom Express    12610
Shadowfax        5816
FedEx India      5415
DHL Express      2747
Name: count, dtype: int64

In [28]:
def predict_next_booking_date(company_df: pd.DataFrame) -> tuple:
    unique_dates = pd.to_datetime(sorted(company_df['booking_date'].unique()))
    last_date    = unique_dates[-1]

    gaps         = pd.Series(unique_dates).diff().dt.days.dropna()
    recent_gaps  = gaps.iloc[-30:]

    weights          = np.arange(1, len(recent_gaps) + 1)
    weighted_avg_gap = np.average(recent_gaps, weights=weights)

    gap_days       = max(1, round(weighted_avg_gap))
    predicted_date = last_date + timedelta(days=gap_days)

    return predicted_date, round(weighted_avg_gap, 2)

In [29]:
def predict_shipment_type(company_df: pd.DataFrame, last_date: pd.Timestamp) -> tuple:
    recent_cutoff = last_date - timedelta(days=90)
    recent_df     = company_df[company_df['booking_date'] >= recent_cutoff]

    if len(recent_df) == 0:
        recent_df = company_df

    type_counts = recent_df['shipment_type'].value_counts()
    total       = type_counts.sum()

    predicted_type = type_counts.index[0]
    confidence     = round(type_counts.iloc[0] / total * 100, 1)

    if len(type_counts) > 1:
        second_type = type_counts.index[1]
        second_conf = round(type_counts.iloc[1] / total * 100, 1)
    else:
        second_type = predicted_type
        second_conf = 0.0

    return predicted_type, confidence, second_type, second_conf

In [30]:
records = []

for company in sorted(df['company_name'].unique()):
    company_df = df[df['company_name'] == company].sort_values('booking_date')
    last_date  = company_df['booking_date'].max()

    next_date, avg_gap                          = predict_next_booking_date(company_df)
    pred_type, confidence, second_type, second_conf = predict_shipment_type(company_df, last_date)

    records.append({
        'Company'                : company,
        'Last Booking Date'      : str(last_date.date()),
        'Avg Gap (days)'         : avg_gap,
        'Predicted Next Date'    : str(next_date.date()),
        'Predicted Shipment Type': pred_type,
        'Type Confidence (%)'    : confidence,
        'Alt Shipment Type'      : second_type,
        'Alt Confidence (%)'     : second_conf,
        'Total Bookings'         : len(company_df),
    })

results_df = pd.DataFrame(records)
results_df

,Company,Last Booking Date,Avg Gap (days),Predicted Next Date,Predicted Shipment Type,Type Confidence (%),Alt Shipment Type,Alt Confidence (%),Total Bookings
0,BlueDart,2025-12-31,1.20,2026-01-01,Surface,44.1,Air,29.1,31636
1,DHL Express,2025-12-31,2.39,2026-01-02,Surface,44.8,Air,31.5,2747
2,DTDC,2025-12-31,1.29,2026-01-01,Surface,43.0,Air,32.2,20576
3,Delhivery,2025-12-31,1.42,2026-01-01,Surface,45.3,Air,29.0,33454
4,Ecom Express,2025-12-31,1.45,2026-01-01,Surface,43.2,Air,30.9,12610
5,FedEx India,2025-12-26,2.46,2025-12-28,Surface,41.0,Air,34.6,5415
6,Shadowfax,2025-12-30,2.72,2026-01-02,Surface,47.8,Air,27.8,5816
7,XpressBees,2025-12-31,1.63,2026-01-02,Surface,44.3,Air,30.7,16566


In [31]:
results_df[['Company', 'Predicted Next Date', 'Predicted Shipment Type', 'Type Confidence (%)']].set_index('Company')

,Predicted Next Date,Predicted Shipment Type,Type Confidence (%)
Company,,,
BlueDart,2026-01-01,Surface,44.1
DHL Express,2026-01-02,Surface,44.8
DTDC,2026-01-01,Surface,43.0
Delhivery,2026-01-01,Surface,45.3
Ecom Express,2026-01-01,Surface,43.2
FedEx India,2025-12-28,Surface,41.0
Shadowfax,2026-01-02,Surface,47.8
XpressBees,2026-01-02,Surface,44.3


In [32]:
results_df.to_csv('initial_time-series_shipment_predictions_output.csv', index=False)